<h2>Content-Based Movie Recommendation System</h2>

<p style="font-size:16px;">I am creating this Movie Recommendation Model for my Netflix Web App. The objective of this notebook is to create a reliable Content-based movie recommendation model using the TMDB 5000 Movie Dataset. To achieve this purpose, I will be experimenting with various statistical and machine learning methods.</p>

<h5>Table of Contents</h5>
<ul type='o'>
    <li><a href="#DataCleaning">Data Cleaning</a>
    <ol>
        <li>Extracting Relevant Crew Details and Creating separate columns.</li>
        <li>Extracting 10 Cast Names</li>
        <li>Cleaning 'genre', 'keywords', 'languages', and other columns.</li>
        <li>Merging the dataset</li>
        <li>Removing unnecessary Columns</li>
    </ol></li>
    <li><a href="#DataCleaning">Data Preprocessing</a><ol>
        <li>Cleaning the text columns using Regex.</li>
        <li>Combining all the text into 'Combined_text' column.</li>
        <li>Normalisation and Scaling of Numerical features.</li>
        <li>Vectorize the text using TF-IDF Vectorizer.</li>
        <li>Creating CSR Matrix and Stacking the features.</li>
    </ol></li>
    <li><a href="#DataCleaning">Model Building</a><br><ol>
        <li>Cosine Similarity</li>
        <li>Euclidean Distance</li>
        <li>KNearestNeighbours</li>
    </ol></li>
    <li><a href="#DataCleaning">Using Sentence Transformers Embeddings</a><ol>
        <li>Creating Embedding using Model</li>
        <li>Training on KNN algorithm and testing recommendations.</li>
    </ol></li>
</ul>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv
/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv


In [2]:
pd.set_option('display.max_colwidth', None)

In [3]:
movies_df = pd.read_csv('/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv')

In [4]:
credits_df = pd.read_csv('/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv')

In [5]:
movies_df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 878, ""name"": ""Science Fiction""}]",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id"": 3386, ""name"": ""space war""}, {""id"": 3388, ""name"": ""space colony""}, {""id"": 3679, ""name"": ""society""}, {""id"": 3801, ""name"": ""space travel""}, {""id"": 9685, ""name"": ""futuristic""}, {""id"": 9840, ""name"": ""romance""}, {""id"": 9882, ""name"": ""space""}, {""id"": 9951, ""name"": ""alien""}, {""id"": 10148, ""name"": ""tribe""}, {""id"": 10158, ""name"": ""alien planet""}, {""id"": 10987, ""name"": ""cgi""}, {""id"": 11399, ""name"": ""marine""}, {""id"": 13065, ""name"": ""soldier""}, {""id"": 14643, ""name"": ""battle""}, {""id"": 14720, ""name"": ""love affair""}, {""id"": 165431, ""name"": ""anti war""}, {""id"": 193554, ""name"": ""power relations""}, {""id"": 206690, ""name"": ""mind and soul""}, {""id"": 209714, ""name"": ""3d""}]",en,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289}, {""name"": ""Twentieth Century Fox Film Corporation"", ""id"": 306}, {""name"": ""Dune Entertainment"", ""id"": 444}, {""name"": ""Lightstorm Entertainment"", ""id"": 574}]","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}, {""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""}]",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso_639_1"": ""es"", ""name"": ""Espa\u00f1ol""}]",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""name"": ""Action""}]",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""name"": ""drug abuse""}, {""id"": 911, ""name"": ""exotic island""}, {""id"": 1319, ""name"": ""east india trading company""}, {""id"": 2038, ""name"": ""love of one's life""}, {""id"": 2052, ""name"": ""traitor""}, {""id"": 2580, ""name"": ""shipwreck""}, {""id"": 2660, ""name"": ""strong woman""}, {""id"": 3799, ""name"": ""ship""}, {""id"": 5740, ""name"": ""alliance""}, {""id"": 5941, ""name"": ""calypso""}, {""id"": 6155, ""name"": ""afterlife""}, {""id"": 6211, ""name"": ""fighter""}, {""id"": 12988, ""name"": ""pirate""}, {""id"": 157186, ""name"": ""swashbuckler""}, {""id"": 179430, ""name"": ""aftercreditsstinger""}]",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But nothing is quite as it seems.",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""name"": ""Jerry Bruckheimer Films"", ""id"": 130}, {""name"": ""Second Mate Productions"", ""id"": 19936}]","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}]",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 80, ""name"": ""Crime""}]",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name"": ""based on novel""}, {""id"": 4289, ""name"": ""secret agent""}, {""id"": 9663, ""name"": ""sequel""}, {""id"": 14555, ""name"": ""mi6""}, {""id"": 156095, ""name"": ""british secret service""}, {""id"": 158431, ""n

In [6]:
credits_df.head(1)

movie_id   title  \
0     19995  Avatar   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

<h3 id="DataCleaning">Data Cleaning</h3>

<p>As we can see, there are 100's of crew members working for a movie. Using all the crew information in a movie recommendation model seems impractical due to the sheer volume of data and for the fact that only few top level position such as Director, Writer, etc would be significant for the model.</p>

<p>Im only including the following crew designation's for the training data - Director, Producer, Screenplay, Writer, Music Director, Production Designer, Art Director, Casting]</p>

Steps - 1. Keep only relevant crew member
2. Keep only first 10 cast. 
3. Extract all the production company names. 
4. Extract all the genres names.
5. First 10 keywords.
6 Extract country and language used Iso's. 

<

In [7]:
credits_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4803 non-null   int64 
 1   title     4803 non-null   object
 2   cast      4803 non-null   object
 3   crew      4803 non-null   object
dtypes: int64(1), object(3)
memory usage: 150.2+ KB


In [8]:
credits_df.isnull().sum()

movie_id    0
title       0
cast        0
crew        0
dtype: int64

In [9]:
credits_df.duplicated().sum()

np.int64(0)

In [10]:
import ast

# Convert string representation to actual list/dict
credits_df["crew"] = credits_df["crew"].apply(ast.literal_eval)

In [11]:
credits_df.head(1)

movie_id   title  \
0     19995  Avatar   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [12]:
#function to extract relevant crew names based on thier job titles
def extract_relevant_crew(crew_list, job):
    for item in crew_list:
        if item["job"] == job:
            return item["name"]
    return None

In [13]:
type(credits_df['crew'])

pandas.core.series.Series

In [14]:
# extracting relevant crew titles and organizing into distinct columns
relevant_crew = ['Director', 'Producer', 'Screenplay', 'Casting', 'Original Music Composer', 'Production Design', 'Co-Producer', 'Editor']
for i in relevant_crew:
    credits_df[i] = credits_df["crew"].apply(lambda x: extract_relevant_crew(x, str(i)))

In [15]:
# Convert string to actual list
credits_df["cast"] = credits_df["cast"].apply(ast.literal_eval)

# Function to extract first 10 cast members
def extract_cast(cast_list):
    result = []
    
    for item in cast_list[:10]:
        result.append({
            item["name"]: item["character"]
        })
    
    return result

# Applying
credits_df["top_cast"] = credits_df["cast"].apply(extract_cast)

print(credits_df["top_cast"].iloc[0])

[{'Sam Worthington': 'Jake Sully'}, {'Zoe Saldana': 'Neytiri'}, {'Sigourney Weaver': 'Dr. Grace Augustine'}, {'Stephen Lang': 'Col. Quaritch'}, {'Michelle Rodriguez': 'Trudy Chacon'}, {'Giovanni Ribisi': 'Selfridge'}, {'Joel David Moore': 'Norm Spellman'}, {'CCH Pounder': 'Moat'}, {'Wes Studi': 'Eytukan'}, {'Laz Alonso': "Tsu'Tey"}]


In [16]:
cleaned_credits = credits_df.copy()

In [17]:
# Removing old cast and crew columns 
cleaned_credits.drop(['cast','crew'], axis=1, inplace=True)

In [18]:
movies_df.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 878, ""name"": ""Science Fiction""}]",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id"": 3386, ""name"": ""space war""}, {""id"": 3388, ""name"": ""space colony""}, {""id"": 3679, ""name"": ""society""}, {""id"": 3801, ""name"": ""space travel""}, {""id"": 9685, ""name"": ""futuristic""}, {""id"": 9840, ""name"": ""romance""}, {""id"": 9882, ""name"": ""space""}, {""id"": 9951, ""name"": ""alien""}, {""id"": 10148, ""name"": ""tribe""}, {""id"": 10158, ""name"": ""alien planet""}, {""id"": 10987, ""name"": ""cgi""}, {""id"": 11399, ""name"": ""marine""}, {""id"": 13065, ""name"": ""soldier""}, {""id"": 14643, ""name"": ""battle""}, {""id"": 14720, ""name"": ""love affair""}, {""id"": 165431, ""name"": ""anti war""}, {""id"": 193554, ""name"": ""power relations""}, {""id"": 206690, ""name"": ""mind and soul""}, {""id"": 209714, ""name"": ""3d""}]",en,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289}, {""name"": ""Twentieth Century Fox Film Corporation"", ""id"": 306}, {""name"": ""Dune Entertainment"", ""id"": 444}, {""name"": ""Lightstorm Entertainment"", ""id"": 574}]","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}, {""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""}]",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso_639_1"": ""es"", ""name"": ""Espa\u00f1ol""}]",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [19]:
# function to extract all the 'name' keys from dictionaries
def extract_names(names_list):
    result = []
    for item in names_list:
        result.append(item["name"])
    return result

In [20]:
movies_df['genres'] = movies_df['genres'].apply(ast.literal_eval)

In [21]:
movies_df['genres'] = movies_df['genres'].apply(extract_names)

In [22]:
movies_df['keywords'] = movies_df['keywords'].apply(ast.literal_eval)

In [23]:
movies_df['spoken_languages'] = movies_df['spoken_languages'].apply(ast.literal_eval)
movies_df['production_companies'] = movies_df['production_companies'].apply(ast.literal_eval)
movies_df['production_countries'] = movies_df['production_countries'].apply(ast.literal_eval)

In [24]:
movies_df['keywords'] = movies_df['keywords'].apply(extract_names)
movies_df['production_companies'] = movies_df['production_companies'].apply(extract_names)

In [25]:
#function to extract iso abbreviation from columns
def extract_iso(iso_list, iso):
    result = []
    for item in iso_list:
        result.append(item[iso])
    return result

In [26]:
movies_df['spoken_languages'] = movies_df['spoken_languages'].apply(lambda x: extract_iso(x, "iso_639_1"))
movies_df['production_countries'] = movies_df['production_countries'].apply(lambda x: extract_iso(x, "iso_3166_1"))

In [27]:
movies_df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[Action, Adventure, Fantasy, Science Fiction]",http://www.avatarmovie.com/,19995,"[culture clash, future, space war, space colony, society, space travel, futuristic, romance, space, alien, tribe, alien planet, cgi, marine, soldier, battle, love affair, anti war, power relations, mind and soul, 3d]",en,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.",150.437577,"[Ingenious Film Partners, Twentieth Century Fox Film Corporation, Dune Entertainment, Lightstorm Entertainment]","[US, GB]",2009-12-10,2787965087,162.0,"[en, es]",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[Adventure, Fantasy, Action]",http://disney.go.com/disneypictures/pirates/,285,"[ocean, drug abuse, exotic island, east india trading company, love of one's life, traitor, shipwreck, strong woman, ship, alliance, calypso, afterlife, fighter, pirate, swashbuckler, aftercreditsstinger]",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But nothing is quite as it seems.",139.082615,"[Walt Disney Pictures, Jerry Bruckheimer Films, Second Mate Productions]",[US],2007-05-19,961000000,169.0,[en],Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[Action, Adventure, Crime]",http://www.sonypictures.com/movies/spectre/,206647,"[spy, based on novel, secret agent, sequel, mi6, british secret service, united kingdom]",en,Spectre,"A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles political forces to keep the secret service alive, Bond peels back the layers of deceit to reveal the terrible truth behind SPECTRE.",107.376788,"[Columbia Pictures, Danjaq, B24]","[GB, US]",2015-10-26,880674609,148.0,"[fr, en, es, it, de]",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[Action, Crime, Drama, Thriller]",http://www.thedarkknightrises.com/,49026,"[dc comics, crime fighter, terrorist, secret identity, burglar, hostage drama, time bomb, gotham city, vigilante, cover-up, superhero, villainess, tragic hero, terrorism, destruction, catwoman, cat burglar, imax, flood, criminal underworld, batman]",en,The Dark Knight Rises,"Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's crimes to protect the late attorney's reputation and is subsequently hunted by the Gotham City Police Department. Eight years later, Batman encounters the mysterious Selina Kyle and the villainous Bane, a new terrorist leader who overwhelms Gotham's finest. The Dark Knight resurfaces to protect a city that has branded him an enemy.",112.312950,"[Legendary Pictures, Warner Bros., DC Entertainment, Syncopy]",[US],2012-07-16,1084939099,165.0,[en],Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[Action, Adventure, Science Fiction]",http://movies.disney.com/john-carter,49529,"[based on novel, mars, medallion, space travel, princess, alien, steampunk, martian, escape, edgar rice burroughs, alien race, superhuman strength, mars civilization, sword and planet, 19th century, 3d]",en,John Carter,"John Carter is a war-weary, former military captain who's inexplicably transported to the mysterious and exotic planet of Barsoom (Mars) and reluctantly becomes embroiled in an epic conflict. It's a world on the brink of collapse, and Carter rediscovers his humanity when he realizes the survival of Barsoom and its people rests in his hands.",43.926995,[Walt Disney Pictures],[US],2012-03-07,284139100,132.0,[en],Released,"

In [28]:
# Merging the two cleaned datasets
movies_df['title'].duplicated().sum()

np.int64(3)

In [29]:
movies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [30]:
movies_df[movies_df[['title', 'release_date']].duplicated(keep=False)]

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count


In [31]:
#merging the dataframes based on id

#Checking for dupliccate ids
print(movies_df['id'].duplicated().sum())
print(cleaned_credits['movie_id'].duplicated().sum())

0
0


In [32]:
# merging the 2 dataframes
full_df = movies_df.merge(cleaned_credits, left_on='id', right_on='movie_id',how='inner')

In [33]:
full_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   budget                   4803 non-null   int64  
 1   genres                   4803 non-null   object 
 2   homepage                 1712 non-null   object 
 3   id                       4803 non-null   int64  
 4   keywords                 4803 non-null   object 
 5   original_language        4803 non-null   object 
 6   original_title           4803 non-null   object 
 7   overview                 4800 non-null   object 
 8   popularity               4803 non-null   float64
 9   production_companies     4803 non-null   object 
 10  production_countries     4803 non-null   object 
 11  release_date             4802 non-null   object 
 12  revenue                  4803 non-null   int64  
 13  runtime                  4801 non-null   float64
 14  spoken_languages        

In [34]:
full_df.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average',
       'vote_count', 'movie_id', 'title_y', 'Director', 'Producer',
       'Screenplay', 'Casting', 'Original Music Composer', 'Production Design',
       'Co-Producer', 'Editor', 'top_cast'],
      dtype='object')

In [35]:
# removing irrelevant columns from the dataframe (tested)
train_df = full_df.drop(['budget','homepage','original_title','status','title_y','vote_average',
                        'vote_count','id','movie_id','production_companies',
       'production_countries','Casting', 'Original Music Composer', 'Production Design',
       'Co-Producer', 'Editor'], axis=1)

In [36]:
train_df.head()

,genres,keywords,original_language,overview,popularity,release_date,revenue,runtime,spoken_languages,tagline,title_x,Director,Producer,Screenplay,top_cast
0,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colony, society, space travel, futuristic, romance, space, alien, tribe, alien planet, cgi, marine, soldier, battle, love affair, anti war, power relations, mind and soul, 3d]",en,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.",150.437577,2009-12-10,2787965087,162.0,"[en, es]",Enter the World of Pandora.,Avatar,James Cameron,James Cameron,James Cameron,"[{'Sam Worthington': 'Jake Sully'}, {'Zoe Saldana': 'Neytiri'}, {'Sigourney Weaver': 'Dr. Grace Augustine'}, {'Stephen Lang': 'Col. Quaritch'}, {'Michelle Rodriguez': 'Trudy Chacon'}, {'Giovanni Ribisi': 'Selfridge'}, {'Joel David Moore': 'Norm Spellman'}, {'CCH Pounder': 'Moat'}, {'Wes Studi': 'Eytukan'}, {'Laz Alonso': 'Tsu'Tey'}]"
1,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india trading company, love of one's life, traitor, shipwreck, strong woman, ship, alliance, calypso, afterlife, fighter, pirate, swashbuckler, aftercreditsstinger]",en,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But nothing is quite as it seems.",139.082615,2007-05-19,961000000,169.0,[en],"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,Gore Verbinski,Jerry Bruckheimer,Ted Elliott,"[{'Johnny Depp': 'Captain Jack Sparrow'}, {'Orlando Bloom': 'Will Turner'}, {'Keira Knightley': 'Elizabeth Swann'}, {'Stellan Skarsgård': 'William ""Bootstrap Bill"" Turner'}, {'Chow Yun-fat': 'Captain Sao Feng'}, {'Bill Nighy': 'Captain Davy Jones'}, {'Geoffrey Rush': 'Captain Hector Barbossa'}, {'Jack Davenport': 'Admiral James Norrington'}, {'Kevin McNally': 'Joshamee Gibbs'}, {'Tom Hollander': 'Lord Cutler Beckett'}]"
2,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi6, british secret service, united kingdom]",en,"A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles political forces to keep the secret service alive, Bond peels back the layers of deceit to reveal the terrible truth behind SPECTRE.",107.376788,2015-10-26,880674609,148.0,"[fr, en, es, it, de]",A Plan No One Escapes,Spectre,Sam Mendes,Barbara Broccoli,John Logan,"[{'Daniel Craig': 'James Bond'}, {'Christoph Waltz': 'Blofeld'}, {'Léa Seydoux': 'Madeleine'}, {'Ralph Fiennes': 'M'}, {'Monica Bellucci': 'Lucia'}, {'Ben Whishaw': 'Q'}, {'Naomie Harris': 'Moneypenny'}, {'Dave Bautista': 'Hinx'}, {'Andrew Scott': 'C'}, {'Rory Kinnear': 'Tanner'}]"
3,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret identity, burglar, hostage drama, time bomb, gotham city, vigilante, cover-up, superhero, villainess, tragic hero, terrorism, destruction, catwoman, cat burglar, imax, flood, criminal underworld, batman]",en,"Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's crimes to protect the late attorney's reputation and is subsequently hunted by the Gotham City Police Department. Eight years later, Batman encounters the mysterious Selina Kyle and the villainous Bane, a new terrorist leader who overwhelms Gotham's finest. The Dark Knight resurfaces to protect a city that has branded him an enemy.",112.312950,2012-07-16,1084939099,165.0,[en],The Legend Ends,The Dark Knight Rises,Christopher Nolan,Charles Roven,Christopher Nolan,"[{'Christian Bale': 'Bruce Wayne / Batman'}, {'Michael Caine': 'Alfred Pennyworth'}, {'Gary Oldman': 'James Gordon'}, {'Anne Hathaway': 'Selina Kyle / Catwoman'}, {'Tom Hardy': 'Bane'}, {'Marion Cotillard': 'Miranda Tate'}, {'Joseph Gordon-Levitt': 'Blake'}, {'Morgan

In [37]:
train_df.columns.to_list()

['genres',
 'keywords',
 'original_language',
 'overview',
 'popularity',
 'release_date',
 'revenue',
 'runtime',
 'spoken_languages',
 'tagline',
 'title_x',
 'Director',
 'Producer',
 'Screenplay',
 'top_cast']

<h3>Data Preprocessing</h3>

In [38]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   genres             4803 non-null   object 
 1   keywords           4803 non-null   object 
 2   original_language  4803 non-null   object 
 3   overview           4800 non-null   object 
 4   popularity         4803 non-null   float64
 5   release_date       4802 non-null   object 
 6   revenue            4803 non-null   int64  
 7   runtime            4801 non-null   float64
 8   spoken_languages   4803 non-null   object 
 9   tagline            3959 non-null   object 
 10  title_x            4803 non-null   object 
 11  Director           4773 non-null   object 
 12  Producer           3780 non-null   object 
 13  Screenplay         2936 non-null   object 
 14  top_cast           4803 non-null   object 
dtypes: float64(2), int64(1), object(12)
memory usage: 563.0+ KB


In [39]:
import re

In [40]:
# function to remove special characters, brackets, colons and convert text to lowercase
def clean_text(x):
    
    # convert to string and lowercase
    x = str(x).lower()
    
    # remove brackets
    x = re.sub(r'[\[\]\(\)\{\}]', ' ', x)
    
    # remove quotes
    x = re.sub(r"[\'\"]", '', x)
    
    # replace commas with spaces
    x = x.replace(',', ' ')

    # replace colons with spaces
    x = x.replace(':', ' ')
    
    # remove extra spaces
    x = re.sub(r'\s+', ' ', x).strip()
    
    return x

In [41]:
# cleaning all the text columns
for col in train_df.select_dtypes(include="object").columns.to_list():
    train_df[col] = train_df[col].apply(clean_text)

In [42]:
train_df.head()

,genres,keywords,original_language,overview,popularity,release_date,revenue,runtime,spoken_languages,tagline,title_x,Director,Producer,Screenplay,top_cast
0,action adventure fantasy science fiction,culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d,en,in the 22nd century a paraplegic marine is dispatched to the moon pandora on a unique mission but becomes torn between following orders and protecting an alien civilization.,150.437577,2009-12-10,2787965087,162.0,en es,enter the world of pandora.,avatar,james cameron,james cameron,james cameron,sam worthington jake sully zoe saldana neytiri sigourney weaver dr. grace augustine stephen lang col. quaritch michelle rodriguez trudy chacon giovanni ribisi selfridge joel david moore norm spellman cch pounder moat wes studi eytukan laz alonso tsutey
1,adventure fantasy action,ocean drug abuse exotic island east india trading company love of ones life traitor shipwreck strong woman ship alliance calypso afterlife fighter pirate swashbuckler aftercreditsstinger,en,captain barbossa long believed to be dead has come back to life and is headed to the edge of the earth with will turner and elizabeth swann. but nothing is quite as it seems.,139.082615,2007-05-19,961000000,169.0,en,at the end of the world the adventure begins.,pirates of the caribbean at worlds end,gore verbinski,jerry bruckheimer,ted elliott,johnny depp captain jack sparrow orlando bloom will turner keira knightley elizabeth swann stellan skarsgård william bootstrap bill turner chow yun-fat captain sao feng bill nighy captain davy jones geoffrey rush captain hector barbossa jack davenport admiral james norrington kevin mcnally joshamee gibbs tom hollander lord cutler beckett
2,action adventure crime,spy based on novel secret agent sequel mi6 british secret service united kingdom,en,a cryptic message from bond’s past sends him on a trail to uncover a sinister organization. while m battles political forces to keep the secret service alive bond peels back the layers of deceit to reveal the terrible truth behind spectre.,107.376788,2015-10-26,880674609,148.0,fr en es it de,a plan no one escapes,spectre,sam mendes,barbara broccoli,john logan,daniel craig james bond christoph waltz blofeld léa seydoux madeleine ralph fiennes m monica bellucci lucia ben whishaw q naomie harris moneypenny dave bautista hinx andrew scott c rory kinnear tanner
3,action crime drama thriller,dc comics crime fighter terrorist secret identity burglar hostage drama time bomb gotham city vigilante cover-up superhero villainess tragic hero terrorism destruction catwoman cat burglar imax flood criminal underworld batman,en,following the death of district attorney harvey dent batman assumes responsibility for dents crimes to protect the late attorneys reputation and is subsequently hunted by the gotham city police department. eight years later batman encounters the mysterious selina kyle and the villainous bane a new terrorist leader who overwhelms gothams finest. the dark knight resurfaces to protect a city that has branded him an enemy.,112.312950,2012-07-16,1084939099,165.0,en,the legend ends,the dark knight rises,christopher nolan,charles roven,christopher nolan,christian bale bruce wayne / batman michael caine alfred pennyworth gary oldman james gordon anne hathaway selina kyle / catwoman tom hardy bane marion cotillard miranda tate joseph gordon-levitt blake morgan freeman lucius fox cillian murphy dr. jonathan crane / scarecrow juno temple jen
4,action adventure science fiction,based on novel mars medallion space travel princess alien steampunk martian escape edgar rice burroughs alien race superhuman strength mars civilization sword and planet 19th century 3d,en,john carter is a war-weary former military captain whos inexplicably transported to the mysterious and exotic planet of barsoom mars and reluctantly beco

In [43]:
# combining all the text columns 
text_cols = train_df.select_dtypes(include="object").columns.to_list()
train_df['combined_text'] = train_df[text_cols].agg(' '.join, axis=1)

In [44]:
train_df['combined_text'].head()

0                                                                                                                                                                                                                                                                                                                             action adventure fantasy science fiction culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d en in the 22nd century a paraplegic marine is dispatched to the moon pandora on a unique mission but becomes torn between following orders and protecting an alien civilization. 2009-12-10 en es enter the world of pandora. avatar james cameron james cameron james cameron sam worthington jake sully zoe saldana neytiri sigourney weaver dr. grace augustine stephen lang col. quaritch michelle rodriguez trudy chacon giovanni ribisi selfridge joel d

In [45]:
# extracting year from date
train_df['release_year'] = pd.to_datetime(
    train_df['release_date'],
    errors='coerce'
).dt.year

In [46]:
# normalization of numerical features
train_df['revenue'] = np.log1p(train_df['revenue'])
train_df['popularity'] = np.log1p(train_df['popularity'])

numeric_cols = [
    'runtime',
    'revenue',
    'release_year'
]

# fill missing values
train_df[numeric_cols] = train_df[numeric_cols].fillna(0)

In [47]:
# applying Standard Scaler to numerical features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_numeric = scaler.fit_transform(train_df[numeric_cols])

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [49]:
from scipy.sparse import hstack, csr_matrix

In [50]:
# vectorizing text column
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1,2)
)

text_vectors = tfidf.fit_transform(train_df['combined_text'])

In [51]:
# stacking both numeric and text features
numeric_sparse = csr_matrix(scaled_numeric)

final_features = hstack([
    text_vectors,
    numeric_sparse
])

In [52]:
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

In [53]:
# cosine similarity model
cosine_sim = cosine_similarity(final_features)

In [54]:
# function to recommend movies using cosine similarity
def recommend_cosine(movie_name, top_n=10):
    
    movie_name = movie_name.lower()

    idx = train_df[train_df['title_x'].str.lower() == movie_name].index[0]
    
    similarity_scores = list(enumerate(cosine_sim[idx]))
    
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )
    
    similarity_scores = similarity_scores[1:top_n+1]
    
    movie_indices = [i[0] for i in similarity_scores]
    
    return train_df['title_x'].iloc[movie_indices]

In [55]:
# euclidean distance model
euclidean_sim = euclidean_distances(final_features)

In [56]:
# function to recommend movies using Euclidean Distance
def recommend_euclidean(movie_name, top_n=10):
    
    movie_name = movie_name.lower()

    idx = train_df[train_df['title_x'].str.lower() == movie_name].index[0]    
    
    distance_scores = list(enumerate(euclidean_sim[idx]))
    
    distance_scores = sorted(
        distance_scores,
        key=lambda x: x[1]
    )
    
    distance_scores = distance_scores[1:top_n+1]
    
    movie_indices = [i[0] for i in distance_scores]
    
    return train_df['title_x'].iloc[movie_indices]

In [57]:
from sklearn.neighbors import NearestNeighbors

In [58]:
# KNearest Neighbour model
knn_model = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

knn_model.fit(final_features)

NearestNeighbors(algorithm='brute', metric='cosine')

In [59]:
# function to recommend movies using knn
def recommend_knn(movie_name, top_n=10):
    
    movie_name = movie_name.lower()

    idx = train_df[train_df['title_x'].str.lower() == movie_name].index[0]    
    
    distances, indices = knn_model.kneighbors(
        final_features[idx],
        n_neighbors=top_n+1
    )
    
    movie_indices = indices.flatten()[1:]
    
    return train_df['title_x'].iloc[movie_indices]


In [60]:

print("\nCOSINE SIMILARITY\n")
print(recommend_cosine("Air Force One"))

print("\nEUCLIDEAN DISTANCE\n")
print(recommend_euclidean("Air Force One"))

print("\nKNN RECOMMENDER\n")
print(recommend_knn("Air Force One"))


COSINE SIMILARITY

214                           the perfect storm
507                            independence day
282                                   true lies
147                             die another day
929                                    outbreak
140                            white house down
809                                forrest gump
279                   terminator 2 judgment day
677                    clear and present danger
233    star wars episode i - the phantom menace
Name: title_x, dtype: object

EUCLIDEAN DISTANCE

944          absolute power
929                outbreak
214       the perfect storm
611    the sum of all fears
573              die hard 2
580      olympus has fallen
422             the 6th day
322       the fifth element
140        white house down
568                     xxx
Name: title_x, dtype: object

KNN RECOMMENDER

214                           the perfect storm
507                            independence day
282                        

As we can see, the recommendations are way off and all the 3 approches - Cosine similarity, Euclidean Distance and KNN; have generated undesired outputs. Research and historical evidence on recommedation systems have shown that these approaches are successful, hence our models have feature-related problems.

I guess the numerical features like popularity score, content non-related columns like Editor, Co-Producer, Production Company and Country are confusing the model and throwing the predictions off grid. 
The solution is to retrain the models after dropping those features.

In [61]:
print("\nCOSINE SIMILARITY\n")
print(recommend_cosine("Toy Story 2"))

print("\nEUCLIDEAN DISTANCE\n")
print(recommend_euclidean("Toy Story 2"))

print("\nKNN RECOMMENDER\n")
print(recommend_knn("Toy Story 2"))


COSINE SIMILARITY

1541                          toy story
42                          toy story 3
494                       the lion king
1062                        a bugs life
1695                            aladdin
812                          pocahontas
194                            dinosaur
231                       monsters inc.
459     spirit stallion of the cimarron
1825           jimmy neutron boy genius
Name: title_x, dtype: object

EUCLIDEAN DISTANCE

1541                      toy story
42                      toy story 3
1062                    a bugs life
533                   monster house
231                   monsters inc.
494                   the lion king
358        atlantis the lost empire
288     the hunchback of notre dame
1695                        aladdin
324           the road to el dorado
Name: title_x, dtype: object

KNN RECOMMENDER

1541                          toy story
42                          toy story 3
494                       the lion king
106

Cosine Similarity and KNN Recommender are providing the exact same results. The recommendations have improved significantly after dropping unnecessary crew and production company columns.

**Embedding based Models**

In [62]:
from sentence_transformers import SentenceTransformer

In [63]:
# creating embeddings using sentence-transformers 
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(
    train_df['combined_text'].tolist(),
    show_progress_bar=True
)


similarity = cosine_similarity(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/151 [00:00<?, ?it/s]

In [64]:
# Knearest Neighbour model
new_nn = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

new_nn.fit(embeddings)

NearestNeighbors(algorithm='brute', metric='cosine')

In [65]:
# function to recommend movies using embedder based KNN
def knn_recommender_embedd(movie_name, top_n=10):

    # find movie index
    matches = train_df[
        train_df['title_x'].str.lower() == movie_name.lower()
    ]

    if matches.empty:
        return "Movie not found"

    idx = matches.index[0]

    # find nearest neighbors
    distances, indices = new_nn.kneighbors(
        [embeddings[idx]],
        n_neighbors=top_n + 1
    )

    # remove first movie (same movie)
    movie_indices = indices.flatten()[1:]

    return train_df['title_x'].iloc[movie_indices]


In [66]:
print(knn_recommender_embedd("Air Force One"))

834        executive decision
140          white house down
1788                  red eye
950            the negotiator
446                   con air
4402      in her line of fire
286                    eraser
2198                  lockout
261     live free or die hard
580        olympus has fallen
Name: title_x, dtype: object


In [67]:
#saving the knn model as a pickle file
import joblib

joblib.dump(
    new_nn,
    '/kaggle/working/knn_model.pkl'
)

['/kaggle/working/knn_model.pkl']

In [68]:
# saving the training df
joblib.dump(
    train_df,
    '/kaggle/working/movies_df.pkl'
)

['/kaggle/working/movies_df.pkl']

In [69]:
# saving the embeddings
np.save(
    '/kaggle/working/embeddings.npy',
    embeddings
)

In [70]:
# saving the movie indices
movie_indices = pd.Series(
    train_df.index,
    index=train_df['title_x'].str.lower()
).drop_duplicates()

joblib.dump(
    movie_indices,
    '/kaggle/working/movie_indices.pkl'
)

['/kaggle/working/movie_indices.pkl']